# Session-based Recommendation with XLNet (Synthetic Data Smoke Test)

Same architecture as the full training notebook, but uses **synthetic data** with a single training loop.
Purpose: quickly verify the training pipeline runs end-to-end in your environment.

In [ ]:
import os
import gc
import math
import datetime
from collections import defaultdict

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR

from transformers import XLNetConfig, XLNetModel
from tqdm.auto import tqdm

device = torch.device("cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Configuration

Feature definitions, model hyperparameters, and training settings — all matching the original T4R notebook.
Synthetic data sizes are configurable below.

In [ ]:
# ---------------------------------------------------------------------------
# Feature configuration
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURES = {
    "sess_pid_seq":  {"cardinality": 164386, "embedding_dim": 64},
    "sess_csid_seq": {"cardinality": 622,    "embedding_dim": 32},
    "sess_ccid_seq": {"cardinality": 129,    "embedding_dim": 16},
    "sess_bid_seq":  {"cardinality": 3426,   "embedding_dim": 32},
}

CONTINUOUS_FEATURES = [
    "sess_price_log_norm_seq",
    "sess_dtime_secs_log_norm_seq",
    "sess_prod_recency_days_log_norm_seq",
]

ITEM_ID_FEATURE = "sess_pid_seq"

# ---------------------------------------------------------------------------
# Model hyperparameters
# ---------------------------------------------------------------------------
MAX_SEQ_LEN = 20
D_MODEL = 448
N_HEAD = 8
N_LAYER = 2
CONTINUOUS_PROJ_DIM = 64          # MLP projection for continuous features
MLM_PROBABILITY = 0.1
LABEL_SMOOTHING = 0.5

# ---------------------------------------------------------------------------
# Training hyperparameters  (same as original notebook)
# ---------------------------------------------------------------------------
LEARNING_RATE = 0.00020171456712823088
WEIGHT_DECAY = 2.747484129693843e-05
NUM_EPOCHS = 10
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 512

# ---------------------------------------------------------------------------
# Synthetic data sizes
# ---------------------------------------------------------------------------
N_TRAIN = 2000
N_VALID = 200
N_TEST = 200

print("Configuration loaded.")

## Synthetic Dataset

Generates random session data matching the schema expected by the model. Sequence lengths are randomised between 3 and `MAX_SEQ_LEN`, with left-padding to `MAX_SEQ_LEN`.

In [ ]:
class SyntheticSessionDataset(Dataset):
    """Generates random sessions matching the schema of the real data."""

    def __init__(self, n_samples, max_seq_len=MAX_SEQ_LEN, seed=42):
        self.n_samples = n_samples
        self.max_seq_len = max_seq_len
        self.rng = np.random.RandomState(seed)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        rng = np.random.RandomState(self.rng.randint(0, 2**31) + idx)
        seq_len = rng.randint(3, self.max_seq_len + 1)
        result = {}

        # --- categorical features: random ids in [1, cardinality), pad with 0
        for feat_name, cfg in CATEGORICAL_FEATURES.items():
            arr = rng.randint(1, cfg["cardinality"], size=seq_len).astype(np.int64)
            padded = np.zeros(self.max_seq_len, dtype=np.int64)
            padded[:seq_len] = arr
            result[feat_name] = torch.tensor(padded, dtype=torch.long)

        # --- continuous features: random normal, pad with 0
        cont_arrays = []
        for _ in CONTINUOUS_FEATURES:
            arr = rng.randn(seq_len).astype(np.float32)
            padded = np.zeros(self.max_seq_len, dtype=np.float32)
            padded[:seq_len] = arr
            cont_arrays.append(padded)
        result["continuous"] = torch.tensor(
            np.stack(cont_arrays, axis=-1), dtype=torch.float32
        )

        # --- attention mask
        mask = np.zeros(self.max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        result["attention_mask"] = torch.tensor(mask, dtype=torch.float32)

        # --- labels
        result["labels"] = result[ITEM_ID_FEATURE].clone()

        return result


# Quick sanity check
_ds = SyntheticSessionDataset(5)
_item = _ds[0]
print(f"Sample dataset size: {len(_ds)}")
for k, v in _item.items():
    print(f"  {k}: {v.shape} {v.dtype}")
del _ds, _item

## Model Architecture

Replicates the Transformers4Rec pipeline:

1. **Categorical embeddings** — one `nn.Embedding` per categorical feature  
2. **Continuous projection** — `Linear(n_continuous, 64) + ReLU`  
3. **Concat aggregation** — concatenate all embeddings → 208 dims  
4. **Input MLP** — project 208 → 448 (`d_model`)  
5. **MLM masking** — replace masked positions with a learned `[MASK]` vector  
6. **XLNet** — `XLNetModel` with bidirectional attention, 2 layers, 8 heads  
7. **Weight-tied output** — project 448 → 64, then `logits = proj @ item_emb.T`

In [ ]:
class SessionXLNetModel(nn.Module):
    """XLNet-based session recommendation model with MLM masking and
    weight-tied next-item prediction head."""

    def __init__(self):
        super().__init__()

        # ---- categorical embeddings (padding_idx=0) --------------------------
        self.embeddings = nn.ModuleDict()
        for feat, cfg in CATEGORICAL_FEATURES.items():
            self.embeddings[feat] = nn.Embedding(
                cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
            )

        # ---- continuous feature projection -----------------------------------
        self.continuous_proj = nn.Sequential(
            nn.Linear(len(CONTINUOUS_FEATURES), CONTINUOUS_PROJ_DIM),
            nn.ReLU(),
        )

        # ---- input MLP: concat_dim → d_model ---------------------------------
        cat_dim = sum(c["embedding_dim"] for c in CATEGORICAL_FEATURES.values())
        total_input_dim = cat_dim + CONTINUOUS_PROJ_DIM  # 64+32+16+32+64 = 208
        self.input_mlp = nn.Sequential(
            nn.Linear(total_input_dim, D_MODEL),
            nn.ReLU(),
        )

        # ---- learned [MASK] token --------------------------------------------
        self.mask_token = nn.Parameter(torch.randn(D_MODEL) * 0.02)

        # ---- XLNet backbone --------------------------------------------------
        xlnet_config = XLNetConfig(
            vocab_size=2,           # unused — we pass inputs_embeds directly
            d_model=D_MODEL,
            n_layer=N_LAYER,
            n_head=N_HEAD,
            d_inner=4 * D_MODEL,
            ff_activation="gelu",
            attn_type="bi",         # bidirectional attention
            mem_len=0,              # no recurrence memory needed
            dropout=0.0,
        )
        self.xlnet = XLNetModel(xlnet_config)

        # ---- weight-tied output head -----------------------------------------
        item_emb_dim = CATEGORICAL_FEATURES[ITEM_ID_FEATURE]["embedding_dim"]
        self.output_proj = nn.Linear(D_MODEL, item_emb_dim, bias=False)
        self.output_bias = nn.Parameter(
            torch.zeros(CATEGORICAL_FEATURES[ITEM_ID_FEATURE]["cardinality"])
        )

        # ---- loss ------------------------------------------------------------
        self.loss_fn = nn.CrossEntropyLoss(
            ignore_index=0, label_smoothing=LABEL_SMOOTHING
        )

    # ------------------------------------------------------------------
    def _item_embedding_weight(self):
        return self.embeddings[ITEM_ID_FEATURE].weight

    # ------------------------------------------------------------------
    def forward(self, batch, training=True):
        # 1. Embed categorical features
        cat_embeds = [self.embeddings[f](batch[f]) for f in CATEGORICAL_FEATURES]

        # 2. Project continuous features
        cont_proj = self.continuous_proj(batch["continuous"])  # (B, L, 64)

        # 3. Concatenate → input MLP
        x = torch.cat(cat_embeds + [cont_proj], dim=-1)       # (B, L, 208)
        x = self.input_mlp(x)                                  # (B, L, 448)

        attention_mask = batch["attention_mask"]                # (B, L)
        labels = batch["labels"]                                # (B, L)

        # 4. MLM masking
        if training:
            rand = torch.rand_like(attention_mask)
            mlm_mask = (rand < MLM_PROBABILITY) & (attention_mask > 0)
        else:
            # Evaluation: mask only the last real position per sequence
            seq_lens = attention_mask.sum(dim=1).long()         # (B,)
            mlm_mask = torch.zeros_like(attention_mask, dtype=torch.bool)
            batch_idx = torch.arange(mlm_mask.size(0), device=mlm_mask.device)
            last_pos = (seq_lens - 1).clamp(min=0)
            mlm_mask[batch_idx, last_pos] = True

        # Replace masked positions with [MASK] token
        mask_expanded = mlm_mask.unsqueeze(-1)                  # (B, L, 1)
        x = torch.where(mask_expanded, self.mask_token.expand_as(x), x)

        # 5. XLNet forward (pass embeddings directly, skip internal word emb)
        xlnet_out = self.xlnet(inputs_embeds=x, attention_mask=attention_mask)
        hidden = xlnet_out.last_hidden_state                    # (B, L, 448)

        # 6. Weight-tied logits
        projected = self.output_proj(hidden)                    # (B, L, 64)
        logits = F.linear(
            projected, self._item_embedding_weight(), self.output_bias
        )                                                       # (B, L, n_items)

        # 7. Loss — only on masked positions (unmasked labels set to 0 → ignored)
        masked_labels = labels.clone()
        masked_labels[~mlm_mask] = 0
        loss = self.loss_fn(
            logits.view(-1, logits.size(-1)), masked_labels.view(-1)
        )

        return loss, logits, mlm_mask


model = SessionXLNetModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## Evaluation Metrics

NDCG@K and Recall@K computed over masked positions (single relevant item per position).

In [ ]:
def compute_ranking_metrics(logits, labels, mlm_mask, top_ks=(20, 40)):
    """Return NDCG@k and Recall@k for the masked positions.

    Args:
        logits:   (B, L, n_items)
        labels:   (B, L)       — ground-truth item ids
        mlm_mask: (B, L) bool  — True at positions we predict
        top_ks:   tuple of k values

    Returns:
        dict  {metric_name: (sum, count)}  — caller accumulates across batches
    """
    masked_logits = logits[mlm_mask]   # (N, n_items)
    masked_labels = labels[mlm_mask]   # (N,)

    if masked_logits.numel() == 0:
        return {}

    results = {}
    for k in top_ks:
        _, topk_idx = torch.topk(masked_logits, k, dim=-1)       # (N, k)
        match = topk_idx == masked_labels.unsqueeze(-1)           # (N, k)

        # Recall@k — fraction of queries where true item is in top-k
        hits = match.any(dim=-1).float()
        results[f"recall_at_{k}"] = (hits.sum().item(), len(hits))

        # NDCG@k — for a single relevant item: DCG = 1/log2(rank+1), IDCG = 1
        # rank is 1-indexed position of the match inside top-k (0 if absent)
        positions = match.float() * torch.arange(
            1, k + 1, device=match.device, dtype=torch.float32
        )
        rank = positions.sum(dim=-1)                              # (N,)
        dcg = torch.where(rank > 0, 1.0 / torch.log2(rank + 1),
                          torch.zeros_like(rank))
        results[f"ndcg_at_{k}"] = (dcg.sum().item(), len(dcg))

    return results

## Training & Evaluation Helpers

In [ ]:
def train_one_day(model, train_loader, optimizer, scheduler, device):
    """Train the model for NUM_EPOCHS over a single day's data."""
    model.train()
    epoch_bar = tqdm(range(NUM_EPOCHS), desc="Epochs", unit="epoch")
    for epoch in epoch_bar:
        epoch_loss = 0.0
        n_batches = 0
        batch_bar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{NUM_EPOCHS}",
                         leave=False, unit="batch")
        for batch in batch_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            loss, _, _ = model(batch, training=True)
            loss.backward()
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            n_batches += 1
            batch_bar.set_postfix(batch_loss=f"{loss.item():.4f}")
        avg_epoch_loss = epoch_loss / max(n_batches, 1)
        epoch_bar.set_postfix(epoch_loss=f"{avg_epoch_loss:.4f}")
        print(f"    Epoch {epoch+1}/{NUM_EPOCHS}  loss={avg_epoch_loss:.4f}")


def evaluate(model, eval_loader, device):
    """Evaluate on masked-last-item prediction, returning averaged metrics."""
    model.eval()
    accum = defaultdict(lambda: [0.0, 0])  # {metric: [sum, count]}
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="  Evaluating", leave=False, unit="batch"):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, logits, mlm_mask = model(batch, training=False)
            total_loss += loss.item()
            n_batches += 1

            metrics = compute_ranking_metrics(
                logits, batch["labels"], mlm_mask
            )
            for k, (s, c) in metrics.items():
                accum[k][0] += s
                accum[k][1] += c

    avg = {k: v[0] / max(v[1], 1) for k, v in accum.items()}
    avg["loss"] = total_loss / max(n_batches, 1)
    return avg


def wipe_memory():
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Training

Single training loop over synthetic data (no sliding window).

In [ ]:
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")

# --- dataloaders (drop_last=False as requested) ---
train_ds = SyntheticSessionDataset(N_TRAIN, seed=42)
valid_ds = SyntheticSessionDataset(N_VALID, seed=123)

train_loader = DataLoader(
    train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=False
)
valid_loader = DataLoader(
    valid_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, drop_last=False
)

# --- optimizer + linear LR schedule ---
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = LambdaLR(
    optimizer, lr_lambda=lambda step: max(0.0, 1 - step / max(total_steps, 1))
)

# --- train ---
train_one_day(model, train_loader, optimizer, scheduler, device)

# --- validate ---
eval_metrics = evaluate(model, valid_loader, device)
print("\nValidation results:")
for k in sorted(eval_metrics):
    print(f"  {k} = {eval_metrics[k]:.6f}")

wipe_memory()
print("\nTraining complete.")

## Test Evaluation

Evaluate the trained model on a synthetic test split.

In [ ]:
test_ds = SyntheticSessionDataset(N_TEST, seed=999)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, drop_last=False)

test_metrics = evaluate(model, test_loader, device)
print("Test results:")
for k in sorted(test_metrics):
    print(f"  {k} = {test_metrics[k]:.6f}")

## Save Model

In [ ]:
model_path = os.path.join(os.getcwd(), "saved_model")
os.makedirs(model_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(model_path, "model.pt"))
print(f"Model saved to {model_path}/model.pt")